# MyPortfolioManagement - Interactive Tutorial

This notebook demonstrates all the key features of the myPortfolioManagement library.

**Contents:**
1. Data Fetching
2. Returns Calculation
3. Portfolio Optimization
4. Performance Metrics
5. Backtesting
6. Clustering
7. Visualization

In [ ]:
# Setup
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from myPortfolioManagement.myReturns import (
    calculate_returns,
    average_returns,
    get_benchmark_returns
)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Data Fetching

Fetch historical price data using Yahoo Finance.

In [ ]:
from myPortfolioManagement.myData import get_stock_prices, get_stock_info

# Define tickers - diversified ETF portfolio
tickers = [
    'SPY',   # S&P 500
    'QQQ',   # Nasdaq 100
    'IWM',   # Small Cap
    'EFA',   # International
    'EEM',   # Emerging Markets
    'AGG',   # Bonds
    'TIP',   # TIPS
    'GLD',   # Gold
    'VNQ',   # Real Estate
]

# Fetch data
prices = get_stock_prices(
    yahoo_tickers=tickers,
    start_date='2018-01-01',
    wide_format=True
)

print(f"Data shape: {prices.shape}")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
prices.head()

In [ ]:
# Get stock information
stock_info = get_stock_info(tickers)
stock_info[['yahooTicker', 'longName', 'sector', 'currency']]

In [ ]:
# Plot normalized prices
normalized = prices / prices.iloc[0] * 100
normalized.plot(figsize=(14, 7), title='Normalized Prices (Base = 100)')
plt.ylabel('Price')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2. Returns Calculation

In [ ]:
# Daily returns
returns = calculate_returns(prices, log_returns=False)
print("Daily Returns Statistics:")
returns.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# Monthly returns
returns_monthly = calculate_returns(prices, convert_to='monthly')
print("Monthly Returns (last 12 months):")
returns_monthly.tail(12)

In [ ]:
# Expected returns (different methods)
mu_hist = average_returns(returns, method='hist', periods=252)
mu_ema = average_returns(returns, method='ema', span=500, periods=252)

expected_returns = pd.DataFrame({
    'Historical': mu_hist,
    'EMA': mu_ema
}).sort_values('Historical', ascending=False)

print("Expected Annual Returns:")
expected_returns.round(4)

In [ ]:
# Get benchmark
benchmark = get_benchmark_returns('S&P500')
print(f"Benchmark shape: {benchmark.shape}")
benchmark.tail()
benchmark

## 3. Portfolio Optimization

In [ ]:
from myPortfolioManagement.myPortfolioOptimisation import (
    HRP,
    equal_weight_portfolio,
    inverse_vol_portfolio,
    port_GMV,
    port_max_sharpe,
    port_CVAR,
    generate_rp_portfolios
)

returns_opt = returns.fillna(0)

In [ ]:
# Equal Weight
w_equal = equal_weight_portfolio(returns_opt)
print("Equal Weight Portfolio:")
w_equal

In [ ]:
# Inverse Volatility
w_inv_vol = inverse_vol_portfolio(returns_opt, weight_max=0.25)
print("Inverse Volatility Portfolio:")
w_inv_vol.sort_values('port_inverse_vol', ascending=False)

In [ ]:
# Hierarchical Risk Parity (HRP)
w_hrp = HRP(
    model='HRP',
    returns_training=returns_opt,
    codependence='pearson',
    rm='MV',
    weight_max=0.25,
    weight_min=0.02
)
print("HRP Portfolio:")
w_hrp.sort_values('port_weight', ascending=False)

In [ ]:
# HERC (Hierarchical Equal Risk Contribution)
w_herc = HRP(
    model='HERC',
    returns_training=returns_opt,
    rm='CVaR',
    weight_max=0.30,
    weight_min=0.02
)
print("HERC Portfolio:")
w_herc.sort_values('port_weight', ascending=False)

In [ ]:
# Global Minimum Variance
w_gmv = port_GMV(returns_opt, weight_min=0.02, weight_max=0.30)
print("Global Minimum Variance Portfolio:")
w_gmv.sort_values('port_min_vol', ascending=False)

In [ ]:
# Maximum Sharpe Ratio
w_sharpe = port_max_sharpe(returns_opt, rf=0.04, weight_min=0.02, weight_max=0.30)
print("Max Sharpe Portfolio:")
w_sharpe.sort_values('port_max_Sharpe', ascending=False)

In [ ]:
# Minimum CVaR
w_cvar = port_CVAR(returns_opt, confidence_interval=0.95, rf=0.04, weight_min=0.02, weight_max=0.30)
print("Minimum CVaR Portfolio:")
w_cvar.sort_values('port_target_CVAR', ascending=False)

In [ ]:
# Compare all portfolios - all functions now return DataFrames with 'asset' as index
all_weights = pd.DataFrame({
    'Equal': w_equal['port_naive'],
    'Inv_Vol': w_inv_vol['port_inverse_vol'],
    'HRP': w_hrp['port_weight'],
    'HERC': w_herc['port_weight'],
    'GMV': w_gmv['port_min_vol'],
    'Max_Sharpe': w_sharpe['port_max_Sharpe'],
    'Min_CVaR': w_cvar['port_target_CVAR']
})

# Validate weights sum to 1
print("Weights sum check:")
print(all_weights.sum().round(4))
print("\nAll Portfolio Weights Comparison:")
all_weights.round(3)

In [ ]:
# Visualize weights
all_weights.plot(kind='bar', figsize=(14, 6), width=0.8)
plt.title('Portfolio Weights Comparison')
plt.xlabel('Asset')
plt.ylabel('Weight')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Performance Metrics

In [ ]:
from myPortfolioManagement.myPerformanceMetrics import (
    get_main_stats,
    alpha_beta_table,
    drawdown_details,
    performance_overview,
    cagr
)
from myPortfolioManagement.myUtils import balance_dates

In [ ]:
# Main performance stats
stats = get_main_stats(returns, rf=0.04)
print("Main Performance Statistics:")
stats.round(4)

In [ ]:
# CAGR
cagr_values = cagr(prices)
print("Compound Annual Growth Rate:")
cagr_values.sort_values(ascending=False).round(4)

In [ ]:
# Alpha-Beta analysis
ab = alpha_beta_table(returns, benchmark, rf=0.04)
print("Alpha-Beta Analysis (Full, Bull, Bear markets):")
ab

In [ ]:
# Drawdown details
spy_prices = prices.iloc[:, 0]
dd = drawdown_details(spy_prices, top_drawdowns=5)
print(f"Top 5 Drawdowns for {prices.columns[0]}:")
dd

## 5. Backtesting

In [ ]:
from myPortfolioManagement.myBacktesting import (
    bootstrap_stats,
    bootstrap_portfolio_performance,
    beating_probability
)

# Create portfolio returns
hrp_weights = w_hrp['port_weight'].to_dict()
portfolio_returns = (returns * pd.Series(hrp_weights)).sum(axis=1)
portfolio_returns.name = 'HRP_Portfolio'

In [ ]:
# Bootstrap statistics
bootstrap_results = bootstrap_stats(
    returns=portfolio_returns,
    returns_benchmark=benchmark.squeeze(),
    rf=0.04,
    n_sim=500
)
print("Bootstrap Statistics:")
bootstrap_results.describe().round(4)

In [ ]:
from myPortfolioManagement.myBacktesting import (
    bootstrap_stats,
    bootstrap_portfolio_performance,
    beating_probability
)

In [ ]:
# Bootstrap with in-sample/out-of-sample
means, dists, dist_stats = bootstrap_portfolio_performance(
    returns=portfolio_returns,
    returns_benchmark=benchmark.squeeze(),
    out_of_sample_date='2023-01-01',
    n_sim=50
)
print("Bootstrap Performance Metrics:")
means.round(4)

In [ ]:
# Probability of beating benchmark
beat_prob = beating_probability(
    returns=pd.DataFrame(portfolio_returns),
    returns_benchmark=benchmark,
    n_sample=50
)
print(f"Probability of beating S&P 500: {beat_prob.values[0][0]:.1%}")

In [ ]:
# Fan Chart - Bootstrap simulation of possible portfolio paths
# Shows confidence intervals for future cumulative returns
from myPortfolioManagement.myBacktesting import fan_chart

fan_chart(
    returns=portfolio_returns,
    out_of_sample_date='2023-01-01',
    n_sample=1000,
    starting_value=1,
    chart_title='HRP Portfolio - Bootstrap Fan Chart'
)

## 6. Clustering

In [ ]:
from myPortfolioManagement.myClustering import (
    ts_clustering,
    cluster_ftca
)

In [ ]:
# Time series clustering
clusters, centers = ts_clustering(
    df=prices,
    number_of_clusters=3,
    algo='dtw',
    plot_bar_center=False
)
print("Asset Clusters:")
clusters

In [ ]:
# FTCA Clustering
ftca = cluster_ftca(returns, threshold=0.5)
print("FTCA Clusters:")
ftca

## 7. Visualization

In [ ]:
from myPortfolioManagement.myPlots import (
    correlation_matrix,
    monthly_heatmap
)

In [ ]:
# Correlation matrix
correlation_matrix(returns, figsize=(10, 8))

In [ ]:
# Monthly heatmap
monthly_heatmap(portfolio_returns, figsize=(12, 8))

In [ ]:
# Portfolio cumulative returns comparison
portfolios = pd.DataFrame({
    'HRP': (returns * pd.Series(w_hrp['port_weight'].to_dict())).sum(axis=1),
    'Equal': (returns * pd.Series(w_equal['port_naive'].to_dict())).sum(axis=1),
    'Max_Sharpe': (returns * pd.Series(w_sharpe['port_max_Sharpe'].to_dict())).sum(axis=1)
})

cumulative = (1 + portfolios).cumprod()
cumulative.plot(figsize=(14, 7), title='Cumulative Returns Comparison')
plt.ylabel('Growth of $1')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Data Fetching**: `get_stock_prices()`, `get_stock_info()`
2. **Returns**: `calculate_returns()`, `average_returns()`
3. **Optimization**: `HRP()`, `port_GMV()`, `port_max_sharpe()`, `port_CVAR()`
4. **Performance**: `get_main_stats()`, `alpha_beta_table()`, `drawdown_details()`
5. **Backtesting**: `bootstrap_stats()`, `beating_probability()`
6. **Clustering**: `ts_clustering()`, `cluster_ftca()`
7. **Visualization**: `correlation_matrix()`, `monthly_heatmap()`